In [0]:
# dbutils.library.restartPython()

In [0]:
from datetime import date
from common_utils.logging import get_logger
from common_utils.ingester import read_jdbc,write_raw
import json

# ======================================
# Logger set up
# ======================================
logger=get_logger('sql_server_ingestion')

# ======================================
# Config Path
# ======================================

dbutils.widgets.text('path',"")
config_path=dbutils.widgets.get('path')
print(config_path)

# ======================================
# Read Config
# ======================================
with open(config_path,"r") as f:
    config=json.load(f)
print(config)

# ======================================
# Config setup
# ======================================

source_config=config['source']
target_config=config['target']
write_options=config['write_options']

run_date=date.today().isoformat()
logger.info('Load date is %s',run_date)


logger.info('Reading data from %s',source_config['dbtable'])

user=dbutils.secrets.get(scope='retail-platform-dev',key=source_config["user"])
password=dbutils.secrets.get(scope='retail-platform-dev',key=source_config["password"])

# ======================================
# Read from sql server
# ======================================
df=read_jdbc(spark,source_config['url'],source_config['dbtable'],user,password)

logger.info('Sample data...')
df.display()
logger.info('In total %s rows are present in the table %s',df.count(),source_config['dbtable'])

target_path=f'{target_config['base_path']}{target_config['folder']}/load_date={run_date}'
write_raw(df, target_path,target_config['file_format'], target_config['mode'], write_options)

logger.info('Data stored at location %s',target_path)


In [0]:
from common_utils.logging import get_logger
from pymongo import MongoClient
from bson import json_util
from common_utils.ingester import read_jdbc,write_raw

logger=get_logger('sql-server-ingestion')

In [0]:
from datetime import date
url = "jdbc:sqlserver://rivadata.database.windows.net:1433;databaseName=batch2;encrypt=true;trustServerCertificate=true;loginTimeout=90" 
user = "rivadata" 
password = "batch2@de"
dbtable='retail.customers'
df=spark.read.format('jdbc')\
    .option('url',url)\
    .option('dbtable',dbtable)\
    .option('user',user)\
    .option('password',password)\
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .load()

run_date=date.today().isoformat()
target_path=f'/Volumes/retaildataplatform/bronze/raw_data/sqlserver_customer/load_date={run_date}'

df.write.mode('overwrite')\
    .option('header','true')\
    .option('quoteAll','True')\
    .option('escape','"').\
    csv(target_path)
df.display()